# 06 - Modelos Predictivos
## Sistema de Predicción Temprana de Plagas - Sierra del Patlachique

**Objetivo:** Entrenar modelos ML para predecir riesgo de plagas

**Responsabilidades:**
- ✓ Dividir datos en train/test
- ✓ Entrenar múltiples modelos (Logística, RandomForest, XGBoost)
- ✓ Validación cruzada
- ✓ Calcular métricas de desempeño
- ✓ Visualizar resultados con Plotly
- ✓ Seleccionar mejor modelo CORRECTAMENTE
- ✓ Hacer predicciones futuras

**Modelos a entrenar:**
1. Regresión Logística - Línea base rápida
2. Random Forest - Captura relaciones no lineales
3. XGBoost - Máximo desempeño

**Entrada:** `data/features/datos_patlachique_features.csv`

**Salida:** 
- Modelos entrenados (.pkl)
- Gráficas de desempeño (HTML)
- Predicciones futuras

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Modelos de ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import xgboost as xgb
import pickle
import os

print("✓ Librerías cargadas correctamente")

# Cargar datos
df = pd.read_csv("../data/features/datos_patlachique_features.csv", 
                  index_col='fecha', parse_dates=True)

print(f"✓ Datos cargados: {df.shape}")

✓ Librerías cargadas correctamente
✓ Datos cargados: (817, 21)


## 1. PREPARACIÓN DE DATOS

**Paso 1: Definir Target y Features**
- TARGET: Indicadores de riesgo (0/1 para cada plaga)
- FEATURES: Variables climáticas + features de ventana móvil

**Paso 2: División Train/Test**
- 80% entrenamiento (histórico)
- 20% prueba (validación)

**Paso 3: Normalización**
- Escalar features para modelos sensibles a escala

In [2]:
print("\n" + "="*70)
print("1. PREPARACIÓN DE DATOS")
print("="*70)

# Definir plagas a predecir
plagas = ['riesgo_chapulin', 'riesgo_pulgon', 'riesgo_cogollero', 'riesgo_arana_roja']

# Features: excluir targets y variables derivadas innecesarias
features_basicas = ['temp_media_C', 'lluvia_mm', 'humedad_suelo_frac', 'humedad_relativa_pct']
features_ventana = ['lluvia_3d', 'lluvia_7d', 'lluvia_14d', 'lluvia_21d',
                    'dias_secos_7d', 'dias_secos_14d',
                    'temp_media_7d', 'temp_media_14d',
                    'hr_media_7d', 'hr_minima_7d', 'dias_hr_baja_14d']

features = features_basicas + features_ventana

print(f"\n📊 DATASET:")
print(f"   Total registros: {len(df)}")
print(f"   Features: {len(features)}")
print(f"   Targets: {len(plagas)} plagas")

# Eliminar filas con NaN (primeras filas por rolling windows)
df_clean = df[features + plagas].dropna()

print(f"\n✓ Datos sin NaN: {len(df_clean)} registros")

# División train/test
X = df_clean[features]
print(f"\n✓ Features X: {X.shape}")

# Guardar para referencia
os.makedirs("../reports", exist_ok=True)
X.describe().round(2).to_csv("../reports/estadisticas_features.csv")
print("✓ Estadísticas guardadas en reports/estadisticas_features.csv")


1. PREPARACIÓN DE DATOS

📊 DATASET:
   Total registros: 817
   Features: 15
   Targets: 4 plagas

✓ Datos sin NaN: 797 registros

✓ Features X: (797, 15)
✓ Estadísticas guardadas en reports/estadisticas_features.csv


## 2. ENTRENAR MODELOS PARA CHAPULÍN

**Estrategia:**
- Entrenar 3 modelos diferentes
- Usar validación cruzada para estimar desempeño real
- Seleccionar el mejor por F1 SCORE (no AUC)

**Métricas:**
- Accuracy: % correcto
- Precision: de los predichos positivos, cuántos son reales
- Recall: de los positivos reales, cuántos detectamos
- F1: balance entre precision y recall
- ROC-AUC: desempeño general

In [3]:
print("\n" + "="*70)
print("2. MODELOS PREDICTIVOS - CHAPULÍN")
print("="*70)

# Target: Chapulín
y_chapulin = df_clean['riesgo_chapulin'].astype(int)

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_chapulin, test_size=0.2, random_state=42, stratify=y_chapulin
)

print(f"\n📊 DIVISIÓN DATOS:")
print(f"   Train: {len(X_train)} muestras")
print(f"   Test: {len(X_test)} muestras")
print(f"   Distribución Train - Positivos: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"   Distribución Test  - Positivos: {y_test.sum()} ({y_test.sum()/len(y_test)*100:.1f}%)")

# Diccionario para guardar resultados
resultados_chapulin = {}

# MODELO 1: Regresión Logística
print("\n🔹 MODELO 1: Regresión Logística")
modelo_lr = LogisticRegression(random_state=42, max_iter=1000)
modelo_lr.fit(X_train, y_train)

y_pred_lr = modelo_lr.predict(X_test)
y_proba_lr = modelo_lr.predict_proba(X_test)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr, zero_division=0)
rec_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
auc_lr = roc_auc_score(y_test, y_proba_lr)

resultados_chapulin['Logística'] = {
    'accuracy': acc_lr, 'precision': prec_lr, 'recall': rec_lr,
    'f1': f1_lr, 'auc': auc_lr, 'modelo': modelo_lr, 'y_proba': y_proba_lr
}

print(f"   Accuracy: {acc_lr:.3f} | Precision: {prec_lr:.3f} | Recall: {rec_lr:.3f} | F1: {f1_lr:.3f} | AUC: {auc_lr:.3f}")

# MODELO 2: Random Forest
print("\n🔹 MODELO 2: Random Forest")
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

y_pred_rf = modelo_rf.predict(X_test)
y_proba_rf = modelo_rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, zero_division=0)
rec_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
auc_rf = roc_auc_score(y_test, y_proba_rf)

resultados_chapulin['RandomForest'] = {
    'accuracy': acc_rf, 'precision': prec_rf, 'recall': rec_rf,
    'f1': f1_rf, 'auc': auc_rf, 'modelo': modelo_rf, 'y_proba': y_proba_rf
}

print(f"   Accuracy: {acc_rf:.3f} | Precision: {prec_rf:.3f} | Recall: {rec_rf:.3f} | F1: {f1_rf:.3f} | AUC: {auc_rf:.3f}")

# MODELO 3: XGBoost
print("\n🔹 MODELO 3: XGBoost")
modelo_xgb = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
modelo_xgb.fit(X_train, y_train)

y_pred_xgb = modelo_xgb.predict(X_test)
y_proba_xgb = modelo_xgb.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb, zero_division=0)
rec_xgb = recall_score(y_test, y_pred_xgb, zero_division=0)
f1_xgb = f1_score(y_test, y_pred_xgb, zero_division=0)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)

resultados_chapulin['XGBoost'] = {
    'accuracy': acc_xgb, 'precision': prec_xgb, 'recall': rec_xgb,
    'f1': f1_xgb, 'auc': auc_xgb, 'modelo': modelo_xgb, 'y_proba': y_proba_xgb
}

print(f"   Accuracy: {acc_xgb:.3f} | Precision: {prec_xgb:.3f} | Recall: {rec_xgb:.3f} | F1: {f1_xgb:.3f} | AUC: {auc_xgb:.3f}")

# ⭐ SELECCIONAR MEJOR MODELO POR F1 SCORE (CORREGIDO)
mejor_modelo_name = max(resultados_chapulin, key=lambda x: resultados_chapulin[x]['f1'])
mejor_f1 = resultados_chapulin[mejor_modelo_name]['f1']
mejor_auc = resultados_chapulin[mejor_modelo_name]['auc']
mejor_recall = resultados_chapulin[mejor_modelo_name]['recall']
mejor_accuracy = resultados_chapulin[mejor_modelo_name]['accuracy']

print(f"\n✅ MEJOR MODELO (por F1 Score): {mejor_modelo_name}")
print(f"   F1 Score: {mejor_f1:.3f} ← MÉTRICA PRINCIPAL")
print(f"   Recall: {mejor_recall:.3f} (detecta {mejor_recall*100:.1f}% de plagas)")
print(f"   Accuracy: {mejor_accuracy:.3f}")
print(f"   AUC: {mejor_auc:.3f}")


2. MODELOS PREDICTIVOS - CHAPULÍN

📊 DIVISIÓN DATOS:
   Train: 637 muestras
   Test: 160 muestras
   Distribución Train - Positivos: 62 (9.7%)
   Distribución Test  - Positivos: 15 (9.4%)

🔹 MODELO 1: Regresión Logística
   Accuracy: 0.894 | Precision: 0.400 | Recall: 0.267 | F1: 0.320 | AUC: 0.924

🔹 MODELO 2: Random Forest
   Accuracy: 0.975 | Precision: 1.000 | Recall: 0.733 | F1: 0.846 | AUC: 0.999

🔹 MODELO 3: XGBoost
   Accuracy: 0.994 | Precision: 1.000 | Recall: 0.933 | F1: 0.966 | AUC: 0.999

✅ MEJOR MODELO (por F1 Score): XGBoost
   F1 Score: 0.966 ← MÉTRICA PRINCIPAL
   Recall: 0.933 (detecta 93.3% de plagas)
   Accuracy: 0.994
   AUC: 0.999


## 3. GRÁFICAS DE DESEMPEÑO - CURVAS ROC

**Visualización:** Curvas ROC para comparar modelos

**Qué muestra:**
- Verdaderos positivos vs Falsos positivos
- Área bajo la curva (AUC) indica desempeño
- Cuanto más arriba y a la izquierda, mejor

In [4]:
print("\n" + "="*70)
print("3. GRÁFICAS DE DESEMPEÑO - CURVAS ROC")
print("="*70)

fig = go.Figure()

# Línea diagonal (clasificador aleatorio)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Aleatorio',
    line=dict(color='gray', dash='dash')
))

# Curva ROC para cada modelo
for nombre, resultado in resultados_chapulin.items():
    fpr, tpr, _ = roc_curve(y_test, resultado['y_proba'])
    auc = resultado['auc']
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{nombre} (AUC={auc:.3f})',
        line=dict(width=2)
    ))

fig.update_layout(
    title='Curvas ROC - Chapulín',
    xaxis_title='Tasa Falsos Positivos',
    yaxis_title='Tasa Verdaderos Positivos',
    hovermode='closest',
    height=500,
    template='plotly_white'
)

fig.show()
os.makedirs("../reports", exist_ok=True)
fig.write_html("../reports/09_roc_chapulin.html")
print("✓ Gráfica guardada: reports/09_roc_chapulin.html")


3. GRÁFICAS DE DESEMPEÑO - CURVAS ROC


✓ Gráfica guardada: reports/09_roc_chapulin.html


## 4. IMPORTANCIA DE FEATURES

**Visualización:** Gráfico de barras con importancia de variables

**Qué muestra:**
- Qué variables son más importantes para predecir
- Helpers para entender qué monitorear
- Identificar variables irrelevantes

In [5]:
print("\n" + "="*70)
print("4. IMPORTANCIA DE FEATURES")
print("="*70)

# Extraer importancias del mejor modelo
modelo_mejor = resultados_chapulin[mejor_modelo_name]['modelo']

# Manejar diferentes tipos de modelos
if mejor_modelo_name == 'Logística':
    importancias = np.abs(modelo_mejor.coef_[0])
    titulo_extra = "(Coeficientes Absolutos)"
else:
    importancias = modelo_mejor.feature_importances_
    titulo_extra = "(Feature Importance)"

indices_sort = np.argsort(importancias)[-15:]  # Top 15

fig = go.Figure(data=[go.Bar(
    y=[features[i] for i in indices_sort],
    x=[importancias[i] for i in indices_sort],
    orientation='h',
    marker=dict(color='#3498DB')
)])

fig.update_layout(
    title=f'Importancia de Features - {mejor_modelo_name} {titulo_extra}',
    xaxis_title='Importancia',
    yaxis_title='Features',
    height=600,
    template='plotly_white'
)

fig.show()
fig.write_html("../reports/10_importancia_features.html")
print("✓ Gráfica guardada: reports/10_importancia_features.html")


4. IMPORTANCIA DE FEATURES


✓ Gráfica guardada: reports/10_importancia_features.html


## 5. MATRIZ DE CONFUSIÓN

**Visualización:** Heatmap de matriz de confusión

**Qué muestra:**
- Verdaderos positivos (TP): correctamente predichos positivos
- Verdaderos negativos (TN): correctamente predichos negativos
- Falsos positivos (FP): falsas alarmas
- Falsos negativos (FN): no detectados

In [6]:
print("\n" + "="*70)
print("5. MATRIZ DE CONFUSIÓN")
print("="*70)

# Usar el mejor modelo
modelo_mejor = resultados_chapulin[mejor_modelo_name]['modelo']
y_pred_mejor = modelo_mejor.predict(X_test)

cm = confusion_matrix(y_test, y_pred_mejor)

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=['Negativo', 'Positivo'],
    y=['Negativo', 'Positivo'],
    text=cm,
    texttemplate='%{text}',
    textfont={"size": 14},
    colorscale='Blues'
))

fig.update_layout(
    title=f'Matriz de Confusión - {mejor_modelo_name}',
    xaxis_title='Predicción',
    yaxis_title='Real',
    height=500,
    width=500
)

fig.show()
fig.write_html("../reports/11_matriz_confusion.html")
print("✓ Gráfica guardada: reports/11_matriz_confusion.html")

# Mostrar métricas detalladas
print(f"\n📊 REPORTE DETALLADO ({mejor_modelo_name}):")
print(classification_report(y_test, y_pred_mejor, target_names=['Sin Riesgo', 'Con Riesgo']))


5. MATRIZ DE CONFUSIÓN


✓ Gráfica guardada: reports/11_matriz_confusion.html

📊 REPORTE DETALLADO (XGBoost):
              precision    recall  f1-score   support

  Sin Riesgo       0.99      1.00      1.00       145
  Con Riesgo       1.00      0.93      0.97        15

    accuracy                           0.99       160
   macro avg       1.00      0.97      0.98       160
weighted avg       0.99      0.99      0.99       160



## 6. COMPARACIÓN DE MODELOS

**Visualización:** Gráfico radar (spider) con todas las métricas

**Qué muestra:**
- Comparación simultánea de Accuracy, Precision, Recall, F1, AUC
- Identifica fortalezas y debilidades de cada modelo

In [7]:
print("\n" + "="*70)
print("6. COMPARACIÓN DE MODELOS")
print("="*70)

# Preparar datos para gráfico radar
categorias = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']

fig = go.Figure()

for nombre, resultado in resultados_chapulin.items():
    valores = [
        resultado['accuracy'],
        resultado['precision'],
        resultado['recall'],
        resultado['f1'],
        resultado['auc']
    ]
    
    fig.add_trace(go.Scatterpolar(
        r=valores,
        theta=categorias,
        fill='toself',
        name=nombre
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    height=600,
    title='Comparación de Modelos - Chapulín'
)

fig.show()
fig.write_html("../reports/12_comparacion_modelos.html")
print("✓ Gráfica guardada: reports/12_comparacion_modelos.html")


6. COMPARACIÓN DE MODELOS


✓ Gráfica guardada: reports/12_comparacion_modelos.html


## 7. GUARDAR MEJOR MODELO

**¿Qué hacemos?**
- Guardar el MEJOR MODELO (ahora correctamente seleccionado)
- Guardar scaler para normalización
- Guardar lista de features

In [8]:
print("\n" + "="*70)
print("7. GUARDAR MODELO ENTRENADO")
print("="*70)

# Crear directorio si no existe
os.makedirs("../models", exist_ok=True)

# Guardar mejor modelo (AHORA CORRECTO)
modelo_final = resultados_chapulin[mejor_modelo_name]['modelo']
archivo_modelo = f"../models/modelo_chapulin_{mejor_modelo_name}.pkl"

with open(archivo_modelo, 'wb') as f:
    pickle.dump(modelo_final, f)

# Guardar scaler
with open("../models/scaler.pkl", 'wb') as f:
    pickle.dump(scaler, f)

# Guardar lista de features
with open("../models/features_list.pkl", 'wb') as f:
    pickle.dump(features, f)

print(f"✓ Modelo {mejor_modelo_name} guardado: {archivo_modelo}")
print(f"✓ Scaler guardado: ../models/scaler.pkl")
print(f"✓ Features guardados: ../models/features_list.pkl")

# Resumen de modelos
print(f"\n📊 RESUMEN DE MODELOS (Chapulín):")
print(f"\n{'Modelo':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'AUC':<10}")
print("-" * 55)
for nombre, resultado in resultados_chapulin.items():
    print(f"{nombre:<15} {resultado['accuracy']:<10.3f} {resultado['precision']:<10.3f} "
          f"{resultado['recall']:<10.3f} {resultado['f1']:<10.3f} {resultado['auc']:<10.3f}")

print(f"\n✅ MODELO GANADOR: {mejor_modelo_name}")


7. GUARDAR MODELO ENTRENADO
✓ Modelo XGBoost guardado: ../models/modelo_chapulin_XGBoost.pkl
✓ Scaler guardado: ../models/scaler.pkl
✓ Features guardados: ../models/features_list.pkl

📊 RESUMEN DE MODELOS (Chapulín):

Modelo          Accuracy   Precision  Recall     F1         AUC       
-------------------------------------------------------
Logística       0.894      0.400      0.267      0.320      0.924     
RandomForest    0.975      1.000      0.733      0.846      0.999     
XGBoost         0.994      1.000      0.933      0.966      0.999     

✅ MODELO GANADOR: XGBoost


## 8. EJEMPLO DE PREDICCIÓN

**¿Qué hacemos?**
Demostrar cómo usar el modelo entrenado para predecir nuevos casos

**Proceso:**
1. Cargar modelo, scaler y features guardados
2. Hacer predicción en últimos 30 días
3. Visualizar probabilidades de riesgo

In [9]:
print("\n" + "="*70)
print("8. EJEMPLO DE PREDICCIÓN CON MODELO")
print("="*70)

# ⭐ CARGAR MODELO CORRECTO (usando variable mejor_modelo_name)
archivo_carga = f"../models/modelo_chapulin_{mejor_modelo_name}.pkl"

with open(archivo_carga, 'rb') as f:
    modelo_demo = pickle.load(f)
with open("../models/scaler.pkl", 'rb') as f:
    scaler_demo = pickle.load(f)
with open("../models/features_list.pkl", 'rb') as f:
    features_demo = pickle.load(f)

print(f"✓ Modelo cargado: {mejor_modelo_name}")
print(f"✓ Archivo: {archivo_carga}")

# Tomar últimos 30 días del dataset
df_ultimos = df_clean[features].tail(30).copy()
X_ultimos_scaled = scaler_demo.transform(df_ultimos)

# Hacer predicciones
predicciones = modelo_demo.predict(X_ultimos_scaled)
probabilidades = modelo_demo.predict_proba(X_ultimos_scaled)[:, 1]

# Crear tabla de predicciones
df_pred = pd.DataFrame({
    'fecha': df_clean.tail(30).index,
    'prediccion': predicciones,
    'probabilidad': probabilidades
})

# Graficar predicciones
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_pred['fecha'],
    y=df_pred['probabilidad'],
    mode='lines+markers',
    name='Probabilidad Riesgo',
    line=dict(color='#E74C3C', width=2),
    marker=dict(size=8)
))

fig.add_hline(y=0.5, line_dash="dash", line_color="red", annotation_text="Umbral (0.5)")

fig.update_layout(
    title=f'Predicciones de Riesgo - Últimos 30 Días ({mejor_modelo_name})',
    xaxis_title='Fecha',
    yaxis_title='Probabilidad de Riesgo',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.show()
fig.write_html("../reports/13_predicciones_futuras.html")
print("✓ Gráfica guardada: reports/13_predicciones_futuras.html")

print("\n" + "="*70)
print("✅ MODELOS PREDICTIVOS COMPLETADOS")
print("="*70)
print(f"""
Se entrenaron 3 modelos para predecir CHAPULÍN:
  ✓ Regresión Logística
  ✓ Random Forest
  ✓ XGBoost

✅ MEJOR MODELO SELECCIONADO: {mejor_modelo_name}
   Métrica: F1 Score = {mejor_f1:.3f}
   Recall: {mejor_recall*100:.1f}% (detecta {mejor_recall*100:.1f}% de las plagas)
   Accuracy: {mejor_accuracy*100:.1f}%
   AUC: {mejor_auc:.3f}

Se generaron 5 gráficas interactivas:
  ✓ 09_roc_chapulin.html
  ✓ 10_importancia_features.html
  ✓ 11_matriz_confusion.html
  ✓ 12_comparacion_modelos.html
  ✓ 13_predicciones_futuras.html

Modelos guardados en: models/
  ✓ modelo_chapulin_{mejor_modelo_name}.pkl (✅ MODELO GANADOR)
  ✓ scaler.pkl
  ✓ features_list.pkl

PRÓXIMOS PASOS:
1. Entrenar modelos para otras plagas (Pulgón, Cogollero, Araña Roja)
2. Automatizar predicciones diarias
3. Crear dashboard con alertas en tiempo real
4. Integrar con sistema de riego/fumigación
""")


8. EJEMPLO DE PREDICCIÓN CON MODELO
✓ Modelo cargado: XGBoost
✓ Archivo: ../models/modelo_chapulin_XGBoost.pkl


✓ Gráfica guardada: reports/13_predicciones_futuras.html

✅ MODELOS PREDICTIVOS COMPLETADOS

Se entrenaron 3 modelos para predecir CHAPULÍN:
  ✓ Regresión Logística
  ✓ Random Forest
  ✓ XGBoost

✅ MEJOR MODELO SELECCIONADO: XGBoost
   Métrica: F1 Score = 0.966
   Recall: 93.3% (detecta 93.3% de las plagas)
   Accuracy: 99.4%
   AUC: 0.999

Se generaron 5 gráficas interactivas:
  ✓ 09_roc_chapulin.html
  ✓ 10_importancia_features.html
  ✓ 11_matriz_confusion.html
  ✓ 12_comparacion_modelos.html
  ✓ 13_predicciones_futuras.html

Modelos guardados en: models/
  ✓ modelo_chapulin_XGBoost.pkl (✅ MODELO GANADOR)
  ✓ scaler.pkl
  ✓ features_list.pkl

PRÓXIMOS PASOS:
1. Entrenar modelos para otras plagas (Pulgón, Cogollero, Araña Roja)
2. Automatizar predicciones diarias
3. Crear dashboard con alertas en tiempo real
4. Integrar con sistema de riego/fumigación

